In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)
random.seed(42)

In [ ]:
pip install mne

In [ ]:
def get_seizure_times(summary_path):
    seizure_dict = {}
    for summary_path in summary_path:

      with open(summary_path, 'r') as f:
          lines = f.readlines()

      current_file = None

      for line in lines:
          line = line.strip()

          if "File Name" in line:
              current_file = line.split(": ")[1]

          if "Seizure Start Time" in line:
              start = int(line.split(": ")[1].split()[0])

          if "Seizure End Time" in line:
              end = int(line.split(": ")[1].split()[0])

              if current_file not in seizure_dict:
                  seizure_dict[current_file] = []

              seizure_dict[current_file].append((start, end))

    return seizure_dict



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder_path = "/content/drive/MyDrive/epilepsy"
summary_path = ["/content/chb01-summary.txt","/content/chb03-summary.txt","/content/chb05-summary.txt"]

In [ ]:

print("Files in folder:")
print(os.listdir(folder_path))

print("\nFiles in summary:")
print(list(get_seizure_times(summary_path).keys()))

Files in folder:
['chb01_01.edf', 'chb01_04.edf.seizures', 'chb01_04.edf', 'chb01_15.edf.seizures', 'chb01_07.edf', 'chb01_15.edf', 'chb01_08.edf', 'chb01_34.edf', 'chb01_18.edf.seizures', 'chb01_21.edf.seizures', 'chb01_18.edf', 'chb01_21.edf', 'chb03_03.edf', 'chb03_07.edf', 'chb03_11.edf', 'chb03_34.edf', 'chb05_13.edf', 'chb05_06.edf']

Files in summary:
['chb01_03.edf', 'chb01_04.edf', 'chb01_15.edf', 'chb01_16.edf', 'chb01_18.edf', 'chb01_21.edf', 'chb01_26.edf', 'chb03_01.edf', 'chb03_02.edf', 'chb03_03.edf', 'chb03_04.edf', 'chb03_34.edf', 'chb03_35.edf', 'chb03_36.edf', 'chb05_06.edf', 'chb05_13.edf', 'chb05_16.edf', 'chb05_17.edf', 'chb05_22.edf']


In [ ]:
seizure_dict = get_seizure_times(summary_path)

In [ ]:
X = []
y = []
file_ids = []

In [ ]:
!pip install numba==0.59.0
# IMPORTANT: After running this cell, please restart the Colab runtime (Runtime -> Restart runtime) and then re-run this cell.

import mne
for file in os.listdir(folder_path):

    if file.endswith(".edf"):

        print("Processing:", file)

        raw = mne.io.read_raw_edf(os.path.join(folder_path, file), preload=True)

        data = raw.get_data()
        sfeq = raw.info['sfreq']

        seizure_times = seizure_dict.get(file, [])

        window_sec=2
        window_size=window_sec*sfeq
        data = raw.get_data()
        num_samples = data.shape[1]

        for start in range(0, num_samples - int(window_size), int(window_size)):
            end = start + int(window_size)

            segment = data[:, start:end]
            label = 0 # Initialize label for each segment

            # Check if this segment overlaps with any seizure time
            for (s, e) in seizure_times:
                start_seizure_sample = int(s * sfeq)
                end_seizure_sample = int(e * sfeq)

                # Check for overlap between current segment and seizure event
                if end > start_seizure_sample and start < end_seizure_sample:
                    label = 1 # Mark as seizure
                    break     # Found an overlap, no need to check other seizure times for this segment

            X.append(segment)
            y.append(label)
            file_ids.append(file)
X = np.array(X)
y = np.array(y)
file_ids = np.array(file_ids)
print("X shape:", X.shape)
print("y shape:", y.shape)
print(num_samples)

Processing: chb01_01.edf
Extracting EDF parameters from /content/drive/MyDrive/epilepsy/chb01_01.edf...
Setting channel info structure...
Creating raw.info structure...


/tmp/ipykernel_5402/3106811148.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(folder_path, file), preload=True)


Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Processing: chb01_04.edf
Extracting EDF parameters from /content/drive/MyDrive/epilepsy/chb01_04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_5402/3106811148.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(folder_path, file), preload=True)


Processing: chb01_07.edf
Extracting EDF parameters from /content/drive/MyDrive/epilepsy/chb01_07.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


...

Processing: chb05_13.edf
Extracting EDF parameters from /content/drive/MyDrive/epilepsy/chb05_13.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_5402/3106811148.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(folder_path, file), preload=True)


Processing: chb05_06.edf
Extracting EDF parameters from /content/drive/MyDrive/epilepsy/chb05_06.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_5402/3106811148.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(folder_path, file), preload=True)


X shape: (25186, 23, 512)
y shape: (25186,)
921600


In [ ]:
import numpy as np

unique, counts = np.unique(y, return_counts=True)

print(dict(zip(unique, counts)))

{0: 24888, 1: 298}


In [ ]:
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import welch

# Configuration
batch_size = 50
n_segments = X.shape[0]
all_features = []

for i in range(0, n_segments, batch_size):
    batch = X[i : i + batch_size]

    mean = np.mean(batch, axis=-1)
    var = np.var(batch, axis=-1)
    rms = np.sqrt(np.mean(batch**2, axis=-1))
    max_val = np.max(batch, axis=-1)
    min_val = np.min(batch, axis=-1)
    sk = skew(batch, axis=-1)
    kurt = kurtosis(batch, axis=-1)

    # 2. Frequency Features
    f, psd = welch(batch, fs=sfeq, axis=-1)
    band_power = np.sum(psd, axis=-1)




    batch_features = np.stack([
        mean, var, rms, sk, kurt, max_val, min_val, band_power
    ], axis=-1)


    all_features.append(batch_features.reshape(batch.shape[0], -1))


X_features = np.concatenate(all_features, axis=0)
print("Final X_features shape:", X_features.shape)

Final X_features shape: (25186, 184)


In [ ]:
import numpy as np

file_ids = np.array(file_ids)

# choose one file for testing
test_file = "chb05_06.edf"

train_idx = file_ids != test_file
test_idx = file_ids == test_file

X_train, X_test = X_features[train_idx], X_features[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", len(y_train))
print("Test size:", len(y_test))
print("Test seizure count:", np.sum(y_test == 1))

Train size: 23387
Test size: 1799
Test seizure count: 58


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
from sklearn.metrics import classification_report


model=RandomForestClassifier(n_estimators=200,random_state=42)
model.fit(X_train,y_train)

y_pred=model.predict(X_test)


print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(f1_score(y_test,y_pred))

print(y_pred)
print(classification_report(y_test, y_pred))

print("Predicted seizures:", np.sum(y_pred == 1))
print("Actual seizures:", np.sum(y_test == 1))

for i in range(len(y_pred)):
    if y_pred[i] == 1:
        print(f"⚠️ Seizure at {i*window_sec}s – {(i+1)*window_sec}s")

0.9883268482490273
0.8936170212765957
0.7241379310344828
0.8
[0 0 0 ... 0 0 0]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1741
           1       0.89      0.72      0.80        58

    accuracy                           0.99      1799
   macro avg       0.94      0.86      0.90      1799
weighted avg       0.99      0.99      0.99      1799

Predicted seizures: 47
Actual seizures: 58
⚠️ Seizure at 438s – 440s
⚠️ Seizure at 440s – 442s
⚠️ Seizure at 442s – 444s
⚠️ Seizure at 454s – 456s
⚠️ Seizure at 456s – 458s
⚠️ Seizure at 458s – 460s
⚠️ Seizure at 460s – 462s
⚠️ Seizure at 462s – 464s
⚠️ Seizure at 464s – 466s
⚠️ Seizure at 466s – 468s
⚠️ Seizure at 468s – 470s
⚠️ Seizure at 470s – 472s
⚠️ Seizure at 472s – 474s
⚠️ Seizure at 474s – 476s
⚠️ Seizure at 476s – 478s
⚠️ Seizure at 478s – 480s
⚠️ Seizure at 480s – 482s
⚠️ Seizure at 482s – 484s
⚠️ Seizure at 484s – 486s
⚠️ Seizure at 486s – 488s
⚠️ Seizure at 488s – 490s
⚠️ Se

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── 1. Build (n_windows, 23) RMS matrix from your existing X and y ──
rms_matrix = np.sqrt(np.mean(X**2, axis=-1))   # shape: (n_windows, 23)

# ── 2. Use only test file windows ──
test_mask   = file_ids == "chb05_06.edf"
rms_test    = rms_matrix[test_mask]             # (1799, 23)
y_test_plot = y[test_mask]

# Clip extreme values so colormap isn't dominated by outliers
rms_test = np.clip(rms_test, 0, np.percentile(rms_test, 99))

# ── 3. Find seizure window indices ──
seizure_idx = np.where(y_test_plot == 1)[0]
seizure_start = seizure_idx[0]
seizure_end   = seizure_idx[-1]

# ── 4. Animation ──
channel_names = [f"CH{i+1}" for i in range(23)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0d0d0d')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d0d0d')

# ── Top: rolling heatmap (show 60-window rolling window) ──
window_view = 60   # how many windows visible at once

im = ax1.imshow(
    rms_test[:window_view].T,
    aspect='auto', cmap='inferno', origin='lower',
    vmin=rms_test.min(), vmax=rms_test.max()
)
ax1.set_yticks(range(23))
ax1.set_yticklabels(channel_names, fontsize=7, color='white')
ax1.set_xlabel("Time (2s windows)", color='white')
ax1.set_title("🧠 EEG Seizure Heatmap — Channel Activation Over Time",
              color='white', fontsize=13, pad=10)
ax1.tick_params(colors='white')
for spine in ax1.spines.values():
    spine.set_edgecolor('#333')

cbar = plt.colorbar(im, ax=ax1, orientation='vertical', pad=0.01)
cbar.set_label('RMS Power', color='white', fontsize=8)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

# Seizure marker line on heatmap
seizure_line = ax1.axvline(x=-1, color='cyan', linewidth=1.5,
                            linestyle='--', alpha=0.0)

time_text = ax1.text(0.01, 0.95, '', transform=ax1.transAxes,
                     color='white', fontsize=9, va='top')
alert_text = ax1.text(0.5, 0.95, '', transform=ax1.transAxes,
                      color='red', fontsize=12, va='top', ha='center',
                      fontweight='bold')

# ── Bottom: mean brain activity line plot ──
mean_activity = rms_test.mean(axis=1)   # (n_windows,)
ax2.set_xlim(0, window_view)
ax2.set_ylim(mean_activity.min() * 0.9, mean_activity.max() * 1.1)
ax2.set_xlabel("Time (2s windows)", color='white')
ax2.set_ylabel("Mean Power", color='white', fontsize=8)
ax2.tick_params(colors='white')
ax2.set_title("Mean Brain Activity", color='white', fontsize=10)
for spine in ax2.spines.values():
    spine.set_edgecolor('#333')

line_plot, = ax2.plot([], [], color='#00ff99', linewidth=1.5)
seizure_line2 = ax2.axvline(x=-1, color='cyan', linewidth=1.5,
                             linestyle='--', alpha=0.0)

# Shade seizure zone on bottom plot
if seizure_start < len(mean_activity):
    ax2.axvspan(0, window_view, alpha=0)   # placeholder, updated in animate

plt.tight_layout(pad=2)

def animate(frame):
    start = frame
    end   = frame + window_view

    # clamp
    end = min(end, len(rms_test))
    chunk = rms_test[start:end]

    # pad if near the end
    if chunk.shape[0] < window_view:
        chunk = np.pad(chunk, ((0, window_view - chunk.shape[0]), (0, 0)))

    im.set_data(chunk.T)
    time_text.set_text(f"t = {start*2}s – {end*2}s")

    # seizure zone highlight
    local_start = seizure_start - frame
    local_end   = seizure_end   - frame

    if 0 <= local_start <= window_view:
        seizure_line.set_xdata([local_start])
        seizure_line.set_alpha(0.9)
        seizure_line2.set_xdata([local_start])
        seizure_line2.set_alpha(0.9)
    else:
        seizure_line.set_alpha(0.0)
        seizure_line2.set_alpha(0.0)

    # alert text
    if seizure_start <= frame + window_view and frame <= seizure_end:
        alert_text.set_text("⚠️  SEIZURE DETECTED")
        fig.patch.set_facecolor('#1a0000')
    else:
        alert_text.set_text("")
        fig.patch.set_facecolor('#0d0d0d')

    # bottom line plot
    x_vals = list(range(min(window_view, end - start)))
    y_vals = mean_activity[start:start + len(x_vals)]
    line_plot.set_data(x_vals, y_vals)

    return [im, seizure_line, seizure_line2,
            time_text, alert_text, line_plot]

n_frames = max(1, len(rms_test) - window_view)
# animate every 5th frame to keep it fast (change to 1 for full detail)
ani = animation.FuncAnimation(fig, animate,
                               frames=range(0, n_frames, 5),
                               interval=80, blit=True)

plt.close()

# ── 5. Save as gif AND display inline ──
ani.save("seizure_heatmap.gif", writer="pillow", fps=15, dpi=80)
print("✅ Saved as seizure_heatmap.gif")
HTML(ani.to_jshtml())